# ONIA · Regresia liniară

Notebook de lucru pentru lecția **2.1. Regresia liniară**. Încearcă să explici fiecare pas înainte de a executa celula.

Concepte: predicții, MSE/MAE, soluția OLS, pseudoinversă, gradient descent, regresie polinomială, Ridge, Lasso, Elastic Net și scikit-learn.

## Predicții pentru câteva valori ale lui x

In [ ]:
w = 2
b = 1

for x in [0, 1, 3, 5]:
    y_estimat = w * x + b
    print(f"x={x}, predicție={y_estimat}")


## Calculul erorii pătratice medii de la zero

In [ ]:
def mse(y_real, y_prezis):
    n = len(y_real)
    suma = 0
    for i in range(n):
        eroare = y_prezis[i] - y_real[i]
        suma += eroare * eroare
    return suma / n

print(mse([2, 4, 6], [3, 2, 7]))  # 2.0


## MAE, MSE și RMSE din liste Python

In [ ]:
from math import sqrt

def metrici(y_real, y_prezis):
    n = len(y_real)
    erori = [y_prezis[i] - y_real[i] for i in range(n)]
    mae = sum(abs(e) for e in erori) / n
    mse = sum(e * e for e in erori) / n
    return mae, mse, sqrt(mse)

print(metrici([3, 4, 5], [4, 2, 5]))
# (1.0, 1.666..., 1.290...)


## Soluția exactă OLS pentru o caracteristică (Python simplu)

In [ ]:
def regresie_ols(x, y):
    n = len(x)
    if n != len(y) or n < 2:
        raise ValueError("Sunt necesare cel puțin două perechi")

    media_x = sum(x) / n
    media_y = sum(y) / n
    sus = sum((x[i] - media_x) * (y[i] - media_y)
              for i in range(n))
    jos = sum((x[i] - media_x) ** 2 for i in range(n))
    if jos == 0:
        raise ValueError("Toate valorile lui x sunt identice")

    w = sus / jos
    b = media_y - w * media_x
    return w, b

print(regresie_ols([1, 2, 3], [2, 3, 5]))
# (1.5, 0.3333333333333335)


## Soluție cu pseudoinversă NumPy

In [ ]:
import numpy as np

# Coloanele 2 și 3 sunt identice.
X = np.array([
    [1, 1, 1],
    [1, 2, 2],
    [1, 3, 3]
], dtype=float)
y = np.array([2, 3, 4], dtype=float)

theta = np.linalg.pinv(X) @ y
print(np.round(theta, 3))  # [1.  0.5 0.5]
print(X @ theta)           # [2. 3. 4.]


## Antrenarea din zero, cu Python de bază

In [ ]:
def antreneaza_linie(x, y, eta=0.05, epoci=2000):
    n = len(x)
    w, b = 0.0, 0.0
    istoric = []

    for epoca in range(epoci):
        gradient_w = 0.0
        gradient_b = 0.0
        eroare_totala = 0.0

        for i in range(n):
            predictie = w * x[i] + b
            eroare = predictie - y[i]
            gradient_w += eroare * x[i]
            gradient_b += eroare
            eroare_totala += eroare ** 2

        gradient_w = 2 * gradient_w / n
        gradient_b = 2 * gradient_b / n
        w -= eta * gradient_w
        b -= eta * gradient_b
        if epoca % 200 == 0:
            istoric.append(eroare_totala / n)

    return w, b, istoric

x = [1, 2, 3]
y = [2, 3, 5]
w, b, erori = antreneaza_linie(x, y, eta=0.05)
print(round(w, 3), round(b, 3))
print([round(e, 3) for e in erori[:4]])


## Regresie cu două sau mai multe caracteristici, de la zero

In [ ]:
def antreneaza(X, y, eta=0.05, epoci=3000):
    n = len(X)
    k = len(X[0])
    w = [0.0] * k
    b = 0.0

    for epoca in range(epoci):
        gw = [0.0] * k
        gb = 0.0
        for i in range(n):
            predictie = b
            for j in range(k):
                predictie += w[j] * X[i][j]
            eroare = predictie - y[i]
            gb += eroare
            for j in range(k):
                gw[j] += eroare * X[i][j]

        b -= eta * (2 * gb / n)
        for j in range(k):
            w[j] -= eta * (2 * gw[j] / n)
    return w, b

X = [[0, 0], [1, 0], [0, 1], [1, 1]]
y = [2, 5, 4, 7]  # y = 3*x1 + 2*x2 + 2
w, b = antreneaza(X, y)
print([round(v, 2) for v in w], round(b, 2))
# [3.0, 2.0] 2.0


## Compararea erorilor de antrenare și validare

In [ ]:
def calculeaza_mse(x, y, w, b):
    total = 0
    for i in range(len(x)):
        eroare = w * x[i] + b - y[i]
        total += eroare * eroare
    return total / len(x)

x_train = [0, 1, 2, 3]
y_train = [1, 3, 5, 7]
x_val = [4, 5]
y_val = [9, 12]

print(calculeaza_mse(x_train, y_train, 2, 1))  # 0.0
print(calculeaza_mse(x_val, y_val, 2, 1))      # 0.5


## Ridge pentru o caracteristică — implementare de la zero

In [ ]:
def ridge_1d(x, y, lam):
    n = len(x)
    mx = sum(x) / n
    my = sum(y) / n
    sxy = sum((x[i] - mx) * (y[i] - my)
              for i in range(n))
    sxx = sum((xi - mx) ** 2 for xi in x)
    if sxx + lam == 0:
        raise ValueError("Nu putem determina o pantă unică")
    w = sxy / (sxx + lam)
    b = my - w * mx
    return w, b

x = [1, 2, 3]
y = [2, 3, 5]
print(ridge_1d(x, y, 0))   # aceeași pantă ca OLS
print(ridge_1d(x, y, 2))   # panta este micșorată


## Lasso unidimensional — soluție exactă, fără biblioteci

In [ ]:
def prag_moale(z, prag):
    if z > prag:
        return z - prag
    if z < -prag:
        return z + prag
    return 0.0

def lasso_1d(x, y, lam):
    n = len(x)
    mx, my = sum(x) / n, sum(y) / n
    sxy = sum((x[i] - mx) * (y[i] - my)
              for i in range(n))
    sxx = sum((xi - mx) ** 2 for xi in x)
    if sxx == 0:
        raise ValueError("Toate valorile lui x sunt identice")
    w = prag_moale(sxy, lam / 2) / sxx
    return w, my - w * mx

print(lasso_1d([1, 2, 3], [2, 3, 5], 10))
# (0.0, 3.3333333333333335)


## Elastic Net cu o singură caracteristică, de la zero

In [ ]:
def prag_moale(z, prag):
    if z > prag:
        return z - prag
    if z < -prag:
        return z + prag
    return 0.0

def elastic_net_1d(x, y, lam1, lam2):
    n = len(x)
    mx = sum(x) / n
    my = sum(y) / n
    sxy = sum((x[i] - mx) * (y[i] - my)
              for i in range(n))
    sxx = sum((xi - mx) ** 2 for xi in x)
    w = prag_moale(sxy, lam1 / 2) / (sxx + lam2)
    b = my - w * mx
    return w, b

print(elastic_net_1d([1, 2, 3], [2, 3, 5], 1, 2))
# (0.625, 2.0833333333333335)


## LinearRegression cu evaluare pe test

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = np.array([[1], [2], [3], [4], [5], [6], [7], [8]], dtype=float)
y = np.array([2.4, 3.1, 4.4, 5.0, 6.2, 7.1, 8.1, 9.4])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Panta:", model.coef_[0])
print("Intercept:", model.intercept_)
print("MAE:", mean_absolute_error(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R²:", r2_score(y_test, pred))
print("Predicția pentru x=9:", model.predict([[9]])[0])


## Comparăm cele patru modele pe aceleași date

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = np.array([[0, 2], [1, 1], [2, 3], [3, 2],
              [4, 5], [5, 3], [6, 6], [7, 4]], dtype=float)
y = np.array([2.1, 3.0, 5.9, 6.8, 10.3, 10.4, 13.9, 14.4])

modele = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.05, max_iter=20000),
    "ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.5,
                             max_iter=20000),
}

for nume, estimator in modele.items():
    # Normalizarea se învață pe setul de antrenare.
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(X, y)
    print(nume, model.predict([[4, 4]])[0])


## Regresie polinomială de gradul 3, cu Ridge

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X = np.linspace(-2, 2, 40).reshape(-1, 1)
y = 1 + 0.5 * X[:, 0] + 2 * X[:, 0] ** 2
y += np.random.default_rng(42).normal(0, 0.3, size=len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    Ridge(alpha=1.0)
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MSE test:", round(mean_squared_error(y_test, pred), 3))


---
# Atelier aplicat · de la intuiție la generalizare

**Obiectivele laboratorului extins:** să poți reproduce matematic o dreaptă de regresie, să explici fiecare reziduu, să implementezi gradient descent, să compari train–validare–test și să recunoști limitele unui model.

> În toate graficele, datele sunt sintetice și servesc demonstrației. Predictibilitatea unei relații nu dovedește cauzalitate.

**Dependențe:** `numpy`, `matplotlib`, `scikit-learn`. Notebookul poate fi rulat în JupyterLab sau Colab. Exemplele sunt deterministe (seed explicit unde este cazul).

## 1 · Panta și interceptul: ce se schimbă pe desen?

Pentru observațiile $(1,2)$, $(2,3)$ și $(3,5)$, desenăm două drepte. O creștere cu o unitate a lui $b$ translatează fiecare predicție cu 1. O creștere a lui $w$ modifică panta: predicțiile diferă din ce în ce mai mult pentru valori mari ale lui $x$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1., 2., 3.])
y = np.array([2., 3., 5.])
xx = np.linspace(0, 4, 150)

fig, ax = plt.subplots(figsize=(8, 4.7))
ax.scatter(x, y, s=80, label="observații", zorder=4)
for w, b, label in [(1, 1, "w=1, b=1"), (1.5, 1/3, "w=1.5, b=1/3 (OLS)"), (1, 2, "w=1, b=2")]:
    ax.plot(xx, w*xx+b, label=label)
ax.set(xlabel="ore de studiu (x)", ylabel="scor (y)", title="Trei drepte, aceleași date")
ax.grid(alpha=.25); ax.legend(); plt.show()


## 2 · Reziduuri, MAE, MSE, RMSE și $R^2$ — calcul desfășurat

Definim $e_i=\hat y_i-y_i$. Atunci

$$\mathrm{MAE}=\frac{1}{n}\sum_i |e_i|,\qquad
\mathrm{MSE}=\frac{1}{n}\sum_i e_i^2,\qquad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}.$$

Pentru $R^2$, comparăm cu predicția constantă $\bar y$:

$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$

Reține: **MSE** are unități la pătrat, **MAE/RMSE** au aceleași unități ca ținta, iar **$R^2$ poate fi negativ**.

In [ ]:
reale = np.array([2., 4., 6., 8.])
prezise = np.array([3., 2., 7., 8.])
erori = prezise - reale
absolut = np.abs(erori)
patrate = erori**2
for i in range(len(reale)):
    print(f"i={i+1}: y={reale[i]:g}, ŷ={prezise[i]:g}, eroare={erori[i]:g}, "
          f"|eroare|={absolut[i]:g}, eroare²={patrate[i]:g}")
mae = absolut.mean()
mse = patrate.mean()
rmse = np.sqrt(mse)
r2 = 1 - patrate.sum() / np.sum((reale - reale.mean())**2)
print(f"MAE={mae:.3f}, MSE={mse:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}")
assert np.isclose(mae, 1.) and np.isclose(mse, 1.5) and np.isclose(r2, .7)


## 3 · Cât de sensibil este MSE la o eroare extremă?

Înlocuim **numai ultima** predicție cu 14, păstrând valorile reale. Comparăm două seturi de valori, pentru a vedea de ce MSE penalizează puternic erorile mari.

In [ ]:
variante = {"obișnuit": np.array([3., 2., 7., 8.]),
            "cu o eroare mare": np.array([3., 2., 7., 14.])}
for nume, estimari in variante.items():
    e = estimari - reale
    ma = np.mean(np.abs(e))
    ms = np.mean(e**2)
    rr = 1 - np.sum(e**2) / np.sum((reale-reale.mean())**2)
    print(f"{nume:18s} · MAE={ma:.2f} · MSE={ms:.2f} · R²={rr:.2f}")


## 4 · OLS — formula pentru o caracteristică, verificată numeric

Panta este $w=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2}$, iar interceptul $b=\bar y-w\bar x$.

*Exercițiu:* înainte de execuție, calculează $\bar x$, $\bar y$, numărătorul și numitorul. Obții $w=1{,}5$ și $b=1/3$?

In [ ]:
x = np.array([1., 2., 3.]); y = np.array([2., 3., 5.])
mx, my = x.mean(), y.mean()
Sxy = np.sum((x-mx)*(y-my))
Sxx = np.sum((x-mx)**2)
w = Sxy / Sxx
b0 = my - w*mx
print(f"x̄={mx:.3f}, ȳ={my:.3f}, Sxy={Sxy:.3f}, Sxx={Sxx:.3f}")
print(f"ŷ = {w:.3f}·x + {b0:.6f}")
assert np.isclose(w,1.5) and np.isclose(b0,1/3)


## 5 · Gradient descent, fiecare pas înregistrat

Pentru $J(w,b)=\frac1n\sum_i(wx_i+b-y_i)^2$, derivatele sunt

$$\frac{\partial J}{\partial w}=\frac2n\sum_i x_i(wx_i+b-y_i),\quad
\frac{\partial J}{\partial b}=\frac2n\sum_i(wx_i+b-y_i).$$

Actualizăm simultan $w \leftarrow w-\eta\,\partial J/\partial w$ și $b\leftarrow b-\eta\,\partial J/\partial b$. Pentru $\eta$ prea mare pot apărea oscilații sau divergență.

In [ ]:
def gradient_descent_demo(xs, ys, eta, epoci=45):
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    w, b = 0.0, 0.0
    urme = []
    for t in range(epoci+1):
        e = w*xs + b - ys
        urme.append((t,w,b,float(np.mean(e**2))))
        gw = 2*np.mean(e*xs)
        gb = 2*np.mean(e)
        w_nou = w - eta*gw
        b_nou = b - eta*gb
        w,b = w_nou,b_nou
    return np.array(urme)

fig, ax = plt.subplots(figsize=(8,4.7))
for eta in [.005,.05,.25]:
    urma = gradient_descent_demo([1,2,3],[2,3,5],eta)
    ax.semilogy(urma[:,0],urma[:,3],label=f"η={eta}")
ax.set(xlabel="iterație",ylabel="MSE (scară logaritmică)",title="Rata de învățare schimbă convergența")
ax.grid(alpha=.25);ax.legend();plt.show()
print("Primul pas cu η=0.05:", gradient_descent_demo([1,2,3],[2,3,5],.05,1)[1])


## 6 · Suprafața costului în spațiul $(w,b)$

Pentru fiecare pereche de coeficienți $(w,b)$ există un MSE. Regresia OLS are o suprafață convexă de tip „bol” pentru costul pătratic. Traseul punctat reprezintă pașii efectuați de gradient descent.

In [ ]:
xx=np.array([1.,2.,3.]); yy=np.array([2.,3.,5.])
Ws=np.linspace(-.4,2.6,110); Bs=np.linspace(-1.0,3.0,110)
WW,BB=np.meshgrid(Ws,Bs)
Z=np.mean((WW[...,None]*xx + BB[...,None]-yy)**2,axis=2)
urma=gradient_descent_demo(xx,yy,.05,70)
fig,ax=plt.subplots(figsize=(7.6,5.2))
c=ax.contourf(WW,BB,np.log1p(Z),levels=18)
fig.colorbar(c,ax=ax,label="log(1 + MSE)")
ax.plot(urma[:,1],urma[:,2],"w.-",ms=3,lw=1,label="gradient descent")
ax.plot([1.5],[1/3],"*",ms=13,label="optim OLS")
ax.set(xlabel="panta w",ylabel="intercept b",title="Coborârea către minim")
ax.legend();plt.show()


## 7 · Regresie polinomială: eroarea de antrenare versus validare

Gradul polinomului controlează flexibilitatea. Un model cu mai mulți coeficienți poate reduce eroarea pe train, dar poate generaliza mai slab. **Alegerea gradului** și a regularizării se face pe validare sau prin cross-validation, nu după testul final.

Folosim `Pipeline`: transformările se potrivesc exclusiv pe datele de train. Datele sunt sintetice, cu o relație adevărată de grad 2.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(42)
X_train = np.linspace(-1,1,21).reshape(-1,1)
X_val = np.linspace(-.97,.97,16).reshape(-1,1)
relatie = lambda x: 1.4+.4*x+1.5*x*x
y_train = relatie(X_train[:,0]) + rng.normal(0,.12,len(X_train))
y_val = relatie(X_val[:,0]) + rng.normal(0,.12,len(X_val))

fig,ax=plt.subplots(figsize=(8,4.8))
ax.scatter(X_train[:,0],y_train,label="antrenare")
ax.scatter(X_val[:,0],y_val,label="validare",marker="x")
T=np.linspace(-1.05,1.05,200).reshape(-1,1)
for grad in (1,2,5,10):
    model = make_pipeline(PolynomialFeatures(degree=grad,include_bias=False),
                          StandardScaler(), Ridge(alpha=.01))
    model.fit(X_train,y_train)
    tr=mean_squared_error(y_train,model.predict(X_train))
    va=mean_squared_error(y_val,model.predict(X_val))
    print(f"grad {grad:2d}: MSE train={tr:.4f}; MSE val={va:.4f}")
    ax.plot(T[:,0],model.predict(T),label=f"grad {grad}")
ax.set(xlabel="x",ylabel="y",title="Train, validare și complexitatea modelului")
ax.grid(alpha=.2);ax.legend();plt.show()


## 8 · Ridge și Lasso: de ce se micșorează coeficienții?

Pentru obiectivul *RSS + $\lambda$·penalizare* și o singură caracteristică centrată, dacă $S_{xy}=30$ și $S_{xx}=10$:

$$w_\text{Ridge}=\frac{30}{10+\lambda},\qquad
w_\text{Lasso}=\frac{\operatorname{sign}(30)\max(|30|-\lambda/2,0)}{10}.$$

Aceste relații folosesc convenția **RSS**, nu scalarea parametrilor `alpha` din scikit-learn. Scopul este să vezi efectul matematic al penalizărilor.

In [ ]:
lambde=np.linspace(0,70,141)
ridge=30/(10+lambde)
lasso=np.maximum(0,(30-lambde/2)/10)
fig,ax=plt.subplots(figsize=(7.7,4.4))
ax.plot(lambde,ridge,label="Ridge (L2)")
ax.plot(lambde,lasso,label="Lasso (L1)")
ax.axhline(3,ls="--",lw=1,label="OLS (λ=0)")
ax.set(xlabel="λ",ylabel="coeficientul w",title="Penalizarea și micșorarea coeficienților")
ax.grid(alpha=.25);ax.legend();plt.show()


## 9 · Diagnosticul reziduurilor și impactul valorilor atipice

Un tipar în reziduuri poate indica o relație ignorată, variații neuniforme ale erorii sau observații atipice. **Nu eliminăm automat un punct** doar fiindcă are reziduu mare; întâi investigăm datele și sensul aplicației.

In [ ]:
from sklearn.linear_model import LinearRegression, HuberRegressor
x=np.arange(1,13,dtype=float).reshape(-1,1)
y=1.5*x[:,0]+1+np.array([.2,-.1,.1,-.2,.05,-.1,.15,.05,-.12,.08,.2,6.0])
ols=LinearRegression().fit(x,y)
robust=HuberRegressor().fit(x,y)
T=np.linspace(0,13,100).reshape(-1,1)
fig,ax=plt.subplots(figsize=(8,4.5))
ax.scatter(x[:,0],y,label="observații")
ax.plot(T[:,0],ols.predict(T),label="OLS")
ax.plot(T[:,0],robust.predict(T),label="Huber (robust)")
ax.set(xlabel="x",ylabel="y",title="O observație atipică poate înclina dreapta")
ax.legend();ax.grid(alpha=.22);plt.show()


## 10 · Train, validare și test: o procedură corectă

1. **Antrenare:** coeficienții se estimează pe train.
2. **Validare:** se aleg hiperparametri (de exemplu gradul sau `alpha`) prin CV pe train.
3. **Test:** se evaluează o singură dată performanța finală pe date nefolosite la selecție.

Pentru scalare și preprocesare, folosim `Pipeline` pentru a preveni scurgerea datelor (*data leakage*).

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng=np.random.default_rng(17)
X=rng.uniform(-2,2,size=(150,1))
y=1.2+.7*X[:,0]+1.1*X[:,0]**2+rng.normal(0,.3,len(X))
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=.2,random_state=7)
pipe=Pipeline([('poly',PolynomialFeatures(include_bias=False)),
               ('scaler',StandardScaler()),('model',Ridge())])
search=GridSearchCV(pipe,{'poly__degree':[1,2,3,5],
                          'model__alpha':[.01,.1,1,10]},cv=5,scoring='neg_mean_squared_error')
search.fit(X_tr,y_tr)
pred=search.predict(X_te)
print('Parametri aleși exclusiv prin CV pe train:',search.best_params_)
print(f'TEST: MAE={mean_absolute_error(y_te,pred):.3f}, '
      f'RMSE={np.sqrt(mean_squared_error(y_te,pred)):.3f}, '
      f'R²={r2_score(y_te,pred):.3f}')


## 11 · Exerciții diferențiate

**Începător**

1. Pentru $\hat y=1{,}5x+1/3$, estimează $y$ la $x=4$ și verifică în Python.
2. Pentru erorile $[-1,0,2]$, determină MAE, MSE și RMSE fără biblioteci.
3. Modifică interceptul dintr-un exemplu și explică grafic ce se întâmplă.

**Intermediar**

4. Calculează manual gradientul pentru punctele $(1,2),(2,3),(3,5)$ pornind din $(w,b)=(0,0)$.
5. Modifică `eta` între 0.005, 0.05 și 0.25. Descrie când eroarea converge sau diverge.
6. Compară gradul 1, 2 și 10 în demonstrația polinomială, urmărind eroarea de validare.

**Avansat**

7. Generează două caracteristici puternic corelate; compară stabilitatea coeficienților OLS și Ridge.
8. Construiește un `Pipeline` cu transformare polinomială, scalare și Ridge. Optimizează gradul și `alpha` prin `GridSearchCV`, apoi evaluează pe test.
9. Explică de ce o regresie polinomială este încă liniară în coeficienți.

> Nu este suficient să obții cifre corecte: pentru fiecare exercițiu, formulează o interpretare în termeni de predicție, reziduu, generalizare sau limită a metodei.

## 12 · Glosar compact

| Termen | Semnificație |
|---|---|
| Caracteristică (`feature`) | Variabilă de intrare, de exemplu numărul de ore de studiu |
| Țintă (`target`) | Valoarea pe care vrem să o aproximăm |
| Coeficient (`weight`) | Număr învățat care influențează predicția |
| Intercept | Predicția modelului pentru intrări zero, când este inclus |
| Reziduu | Diferența dintre predicție și valoarea observată |
| Funcție de cost | Mărime minimizată la antrenare (de exemplu MSE) |
| Rată de învățare | Mărimea pasului în gradient descent |
| Overfitting | Model prea adaptat datelor de train și zgomotului |
| Regularizare | Penalizare pentru a controla coeficienții / flexibilitatea |
| Data leakage | Informație folosită incorect în antrenare, inclusiv din validare/test |

**Ideea de reținut:** definim o predicție → alegem o eroare → optimizăm coeficienții → evaluăm pe date noi.